# Coding Interview Question: Banking Management System

Design and implement a Banking Management System in Python that simulates basic banking operations.

## Problem Statement

Create a console-based application that supports the following features:

- Create a new bank account
- Deposit money
- Withdraw money
- Check account balance
- Transfer money between accounts
- View transaction history

## Account Types

The system should support two types of accounts:

- Savings Account
  - Must maintain a minimum balance
  - Should support interest calculation

- Current Account
  - Should support overdraft facility

## Security and Validation

Implement the following requirements:

- PIN-based authentication
- Private account balance
- Input validation
- Custom exceptions for invalid operations

## Additional Requirements

- Store account data using JSON persistence
- Add logging for important operations
- Provide a menu-driven interface for users

## Expected Outcome

Write clean, object-oriented Python code with proper class design, exception handling, and persistence.

1) Identify Nouns
- Bank
- Account
- SavingsAccount
- CurrentAccount
- Customer
- Transaction

2) Now identify actions
- deposit()
- withdraw()
- transfer()
- calculate_interest()
- authenticate()
- show_transactions()

3) Initial design
                         Account \
                       (Abstract) \
                           │ \
             ┌─────────────┴─────────────┐ \
             │                           │ \
       SavingsAccount              CurrentAccount \
             │                           │ \
       interest                  overdraft_limit 


Customer ──────────────── Account 

Bank \
 │ \
 ├── Customers \
 ├── Accounts \
 └── Transactions

In [34]:
from abc import ABC, abstractmethod
from dataclasses import dataclass
from datetime import datetime

@dataclass
class Transaction:
    transaction_type: str
    amount: float
    timestamp: datetime
    description: str = ""

    def display(self):
        print(
            f"{self.timestamp:%Y-%m-%d %H:%M:%S} | "
            f"{self.transaction_type:<10} | "
            f"₹{self.amount:<10.2f} | "
            f"{self.description}"
        )

class Account(ABC):
    def __init__(self,account_number,owner_name,pin,balance=0):
        self.account_number=account_number
        self.owner_name=owner_name
        self.__pin=pin
        self.__balance=balance
        self.transaction_history=[]

    @property
    def balance(self):
        return self.__balance

    def authenticate(self,pin):
        return self.__pin == pin
    
    def _decrease_balance(self,amount):
        self.__balance -= amount
    
    def _increase_balance(self,amount):
        self.__balance += amount

    def deposit(self,amount,description="Cash Deposit"):
        if amount <= 0:
            raise ValueError("Amount must be greater than 0")
        self._increase_balance(amount)
        transaction = Transaction("Deposit",amount,datetime.now(),description)
        self.transaction_history.append(transaction)

    @abstractmethod
    def withdraw(self,amount):
        pass

    def show_transactions(self):
        if not self.transaction_history:
            print("No transactions found.")
            return
        print(f"\nTransaction History of {self.owner_name}")
        print("-" * 40)
        for transaction in self.transaction_history:
            transaction.display()

class SavingsAccount(Account):
    MIN_BALANCE = 1000
    INTEREST_RATE = 0.04

    def withdraw(self,amount,description="Savings Account Withdrawal"):
        if amount <= 0:
            raise ValueError("Amount must be greater than 0")
        if self.balance - amount < self.MIN_BALANCE:
            raise ValueError("Minimum balance requirement violated")
        self._decrease_balance(amount)
        transaction = Transaction("Withdrawal",amount,datetime.now(),description)
        self.transaction_history.append(transaction)

    def calculate_interest(self):
        return self.__balance * self.INTEREST_RATE

class CurrentAccount(Account):
    def __init__(self,account_number,owner_name,pin,balance=0,overdraft_limit=5000):
        super().__init__(account_number,owner_name,pin,balance)
        self.overdraft_limit = overdraft_limit

    def withdraw(self,amount,description="Current Account Withdrawal"):
        if amount <= 0:
            raise ValueError("Amount must be greater than 0")
        if self.balance - amount < -self.overdraft_limit:
            raise ValueError("Overdraft limit exceeded. Insufficient")
        self._decrease_balance(amount)
        transaction = Transaction("Withdrawal",amount,datetime.now(),description)
        self.transaction_history.append(transaction)

class Bank:
    def __init__(self,name):
        self.name=name
        # Composition:
        # Bank HAS account
        # If bank is closed, all accounts are closed
        self.accounts={}

    def add_account(self,account):
        if account.account_number in self.accounts:
            raise ValueError("Account already exists")

        self.accounts[account.account_number]=account

    def find_account(self,account_number):
        account = self.accounts.get(account_number)

        if not account:
            raise ValueError("Account not found")

        return account

    def transfer(self,source_account_number,target_account_number,amount,pin):
        source_account=self.find_account(source_account_number)
        target_account=self.find_account(target_account_number)

        if not source_account.authenticate(pin):
            raise ValueError("Invalid PIN")

        if amount <= 0:
            raise ValueError("Amount must be greater than 0")

        source_account.withdraw(amount,"Transfer to "+target_account.owner_name)
        target_account.deposit(amount,"Transfer from "+source_account.owner_name)

        print(
            f"₹{amount:.2f} transferred successfully."
        )

    def display_account(self):
        if not self.accounts:
            print("No accounts found.")
            return

        print(f"\nAccounts of {self.name}")
        print("-" * 40)
        for account_number, account in self.accounts.items():
            print(f"Account Number: {account_number}")
            print(f"Owner Name: {account.owner_name}")
            print(f"Balance: ₹{account.balance:.2f}")
            print("-" * 40)


# Usage
# savings_account = SavingsAccount("123456", "John Doe", "1234", 1000)
# savings_account.deposit(500)
# print(savings_account.balance)
# savings_account.withdraw(500)
# print(savings_account.balance)
# savings_account.show_transactions()

# current_account = CurrentAccount("231322", "Asif", "5678", 5000, overdraft_limit=5000)
# current_account.withdraw(9000)
# print(current_account.balance)
# current_account.show_transactions()
# current_account.withdraw(2000)

bank = Bank("Python National Bank")

savings_account = SavingsAccount("123456", "John Doe", "1234", 1000)
current_account = CurrentAccount("231322", "Asif", "5678", 5000, overdraft_limit=5000)

bank.add_account(savings_account)
bank.add_account(current_account)

bank.display_account()

savings_account.deposit(2000,"Salary Deposit")
print("Savings Account Balance:",savings_account.balance)

savings_account.withdraw(1000,"ATM Withdrawal")
print("Savings Account Balance:",savings_account.balance)

bank.transfer("123456","231322",1000,"1234")
print("Transfer between accounts")

bank.display_account()
print("\n--- Savings Transactions ---")
savings_account.show_transactions()
print("\n--- Current Transactions ---")
current_account.show_transactions()






Accounts of Python National Bank
----------------------------------------
Account Number: 123456
Owner Name: John Doe
Balance: ₹1000.00
----------------------------------------
Account Number: 231322
Owner Name: Asif
Balance: ₹5000.00
----------------------------------------
Savings Account Balance: 3000
Savings Account Balance: 2000
₹1000.00 transferred successfully.
Transfer between accounts

Accounts of Python National Bank
----------------------------------------
Account Number: 123456
Owner Name: John Doe
Balance: ₹1000.00
----------------------------------------
Account Number: 231322
Owner Name: Asif
Balance: ₹6000.00
----------------------------------------

--- Savings Transactions ---

Transaction History of John Doe
----------------------------------------
2026-08-09 15:52:12 | Deposit    | ₹2000.00    | Salary Deposit
2026-08-09 15:52:12 | Withdrawal | ₹1000.00    | ATM Withdrawal
2026-08-09 15:52:12 | Withdrawal | ₹1000.00    | Transfer to Asif

--- Current Transactions -